# Data Generation — pool SmartHealth + DirVsAverages from source (same working code)

Each prep cell is the **verbatim** working code (`NewFALVonly.py` / `NewFAIPOnly.py` /
`NewCropPrep.py`) with only two things made configurable at the top:

- **`COHORTS`** — which source folders to pool. Both = combined; comment out `DirVsAverages`
  for the SmartHealth-only set.
- **`CONTRASTS`** (LV/IP only) — which contrasts to save. `[0,1,2,3]` = full (avg/MD/E1/FA);
  **`[0,1]` = the avg+MD MVP** so anyone with plain DWI (no eigenvectors/FA) can reproduce it.
  (Crop is DWI-only already → it is the MVP crop as-is.)

Normalization, mask processing and slice discovery are untouched. Set `datasetname` per run and
run the cell, then its `dataset.json` cell, then copy `DatasetXXX_…` to `nnUNet_raw/` on the server.

### Suggested dataset IDs
| | full 4-contrast | avg+MD MVP |
|---|---|---|
| LV  SmartHealth / combined | 100 / **102** | 140 / **142** |
| IP  SmartHealth / combined | 105 / **107** | 145 / **147** |
| Crop SmartHealth / combined (DWI only) | 110 / **111** | (same 110 / 111) |

## 1. LV — verbatim `NewFALVonly.py` + cohort pool + `CONTRASTS`

In [ ]:



import os
import glob
import re
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Root directory and subfolders
# ============ cohorts to POOL (comment out DirVsAverages for SmartHealth-only) ============
COHORTS = [
    ('/Users/saschastocker/Documents/Stanford/DanEnnis20242025/Paper2025Automatic/Smart_Health',  'Hannum'),
    ('/Users/saschastocker/Documents/Stanford/DanEnnis20242025/Paper2025Automatic/DirVsAverages', 'DirVsAveragesHannum'),
]
OUTPUT_PWD = '/Users/saschastocker/Documents/Stanford/DanEnnis20242025/Paper2025Automatic/Smart_Health'   # where the pooled dataset is written
CONTRASTS = [0, 1, 2, 3]   # 0=avg 1=MD 2=E1 3=FA  ->  use [0, 1] for the avg+MD MVP
datasetname = 'Dataset102_HannumSmartHealthandDirVsAvgs'
output_mask_folder = f'{OUTPUT_PWD}/{datasetname}/labelsTr'
output_image_folder = f'{OUTPUT_PWD}/{datasetname}/imagesTr'
inspection_folder = f'{OUTPUT_PWD}/inspection{datasetname}'

# Ensure output folders exist
os.makedirs(output_mask_folder, exist_ok=True)
os.makedirs(output_image_folder, exist_ok=True)
os.makedirs(inspection_folder, exist_ok=True)

def process_mask_slices(mask_data,lv_only ):
    """
    Combines three 2D mask slices (256, 256 each) into a single mask with:
    - LV (slice 1): 1
    - Insertion point (slice 2): 2
    - Second insertion point (slice 3): 3
    Output is a single 2D mask with shape (256, 256).
    """
    # Initialize the combined mask with zeros (256, 256)
    combined_mask = np.zeros((256, 256), dtype=np.uint8)
    if mask_data.shape[0] == 256 and mask_data.shape[-1] == 3:
        # Reshape to (3, 256, 256)
        mask_data = np.transpose(mask_data, (2, 0, 1))

    # Add slice 1 (LV) with label 1
    
    if(lv_only):
        combined_mask[mask_data[0, :, :] == 1] = 1


    #IPs only!   
    else:
         # Add slice 2 (Insertion point) with label 2
        combined_mask[mask_data[1, :, :] == 1] = 2

        # Add slice 3 (Second insertion point) with label 3
        combined_mask[mask_data[2, :, :] == 1] = 3

    return combined_mask

# Function to normalize images to [0, 1] range
def normalize_image(image):
    image_min = np.min(image)
    image_max = np.max(image)
    return (image - image_min) / (image_max - image_min)

def normalise_MD(image):
    image_min = 0
    image_max = 4
    return (image - image_min) / (image_max - image_min)

def normalise_eigenvector(image):
    # X and Y are components of a UNIT eigenvector (x^2+y^2+z^2=1), so their sum is
    # capped at sqrt(2) (~1.414), not 2 -- the unit-norm constraint couples them.
    # Divide by that fixed sqrt(2) ceiling instead of min-max: every image reaches
    # sqrt(2), whereas a true 0 (a purely through-plane voxel) exists in only ~46%
    # of images, so min-max would anchor on a noise-driven floor. Fixed = stable.
    return image / np.sqrt(2)

# Function to save images for inspection
def save_inspection_plots(image_data, mask_data, filename_base):
    """Saves inspection plots of image, mask, and overlay using matplotlib."""
    
    # Create a figure with 3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Plot 1: Just the image (first channel)
    axes[0].imshow(image_data[:, :, 0], cmap='gray')  # Show first channel (average)
    axes[0].set_title('Average Diffusion Image')
    axes[0].axis('off')

    # Plot 2: Just the mask
    axes[1].imshow(mask_data, cmap='gray')
    axes[1].set_title('Mask Only')
    axes[1].axis('off')

    # Plot 3: Image with mask overlay
    axes[2].imshow(image_data[:, :, 0], cmap='gray')  # Show first channel (average)
    mask_overlay = np.ma.masked_where(mask_data != 1, mask_data)
    axes[2].imshow(mask_overlay, cmap='Reds', alpha=0.5)
    axes[2].set_title('Image with Mask Overlay')
    axes[2].axis('off')

    # Save the figure
    output_file = os.path.join(inspection_folder, f'{filename_base}_inspection.png')
    plt.tight_layout()
    plt.savefig(output_file)
    plt.close(fig)
    print(f'Saved inspection plot: {output_file}')

for pwd, root_folder in COHORTS:
    print(f'root folder: {root_folder}')
    root_path = os.path.join(pwd, root_folder)

    # Loop through volunteer folders
    for volunteer_folder in os.listdir(root_path):
        if volunteer_folder.startswith('Volunteer'):
            volunteer_path = os.path.join(root_path, volunteer_folder)
            distortion_corrected_folder = os.path.join(volunteer_path, 'Distortion_Corrected')

            # Loop through DiVO and MDDW folders
            for divo_folder in os.listdir(distortion_corrected_folder):
                if divo_folder.startswith('DiVO') or divo_folder.startswith('MDDW'):
                    divo_path = os.path.join(distortion_corrected_folder, divo_folder)

                    # Load the quality control information from the Excel file in the DiVO/MDDW folder
                    excel_path = os.path.join(divo_path, 'Detailed_Information.xlsx')
                    if os.path.exists(excel_path):
                        quality_data = pd.read_excel(excel_path)

                        mask_folder = os.path.join(divo_path, '06_Segmentation_Masks_CI')
                        image_folder = os.path.join(divo_path, '05_Segmentation_Images_CI')

                        # Count how many segmentation slices actually exist in this sub-folder
                        # (was hard-coded to the first 3 slices; folders can have many more, e.g. up to 12).
                        slice_files = glob.glob(os.path.join(mask_folder, 'Cropped_Segmentation_Slice_*.nii'))
                        slice_numbers = sorted(
                            int(re.search(r'Slice_(\d+)\.nii$', f).group(1)) for f in slice_files
                        )
                        num_slices = len(slice_numbers)
                        print(f'{volunteer_folder}/{divo_folder}: found {num_slices} slices -> {slice_numbers}')

                        # Loop through every slice that exists in this folder
                        for i in slice_numbers:
                            # Filter rows where "Slice Number" equals i
                            slice_data = quality_data[quality_data['Slice Number'] == i]

                            # Check if we found a matching row and if that row’s image quality is "Good Image Quality"
                            if not slice_data.empty and slice_data.iloc[0]['Image Quality'] == 'Good Image Quality':
                                mask_file = os.path.join(mask_folder, f'Cropped_Segmentation_Slice_{i:03d}.nii')
                                
                            
                                mask_img = nib.load(mask_file)

                                # Assuming the mask file is 3, 256, 256
                                mask_data = mask_img.get_fdata()  # Load the 3D mask (3, 256, 256)

                                # Process and combine the slices
                                combined_mask = process_mask_slices(mask_data,lv_only=True)

                                
                                avg_image_file = os.path.join(image_folder, f'Cropped_Average_Diffusion_Weighted_Image_Slice_{i:03d}.nii')
                                mean_diff_file = os.path.join(image_folder, f'Cropped_Mean_Diffusivty_Image_Slice_{i:03d}.nii')
                                eigenvector_file = os.path.join(image_folder, f'Cropped_Primary_Eigenvector_Image_Slice_{i:03d}.nii')
                                FA_file = os.path.join(image_folder, f'Cropped_Fractional_Anisotropy_Image_Slice_{i:03d}.nii')

                                if os.path.exists(mask_file) and os.path.exists(avg_image_file) and os.path.exists(mean_diff_file) and os.path.exists(eigenvector_file):


                                    # Load the three NIfTI image files (average, mean diffusivity, eigenvector)
                                    avg_img = nib.load(avg_image_file)
                                    mean_diff_img = nib.load(mean_diff_file)
                                    eigenvector_img = nib.load(eigenvector_file)
                                    FA_image = nib.load(FA_file)

                                    # Get the data for all three images
                                    avg_image_data = avg_img.get_fdata()  # (256, 256)
                                    mean_diff_data = mean_diff_img.get_fdata()  # (256, 256)
                                    eigenvector_data = eigenvector_img.get_fdata()  # (256, 256, 3)
                                    FA_image_data = FA_image.get_fdata()

                                    # Normalize each image to [0, 1] range to prevent "washed out" effect
                                    avg_image_data = normalize_image(avg_image_data)
                                    mean_diff_data = normalise_MD(mean_diff_data)
                                    eigenvector_slice1 = eigenvector_data[:, :, 0]
                                    eigenvector_slice2 = eigenvector_data[:, :, 1]

                                    # Combine eigenvector slices 1 and 2 into a single slice
                                    combined_eigenvector_data = eigenvector_slice1 + eigenvector_slice2
                                    combined_eigenvector_data = normalise_eigenvector(combined_eigenvector_data)

                                    # Stack all three channels (average, mean diffusivity, combined eigenvector)
                                    combined_image_data = np.stack([avg_image_data, mean_diff_data, combined_eigenvector_data], axis=-1)

                                    # Save each channel separately (modality files with 0000, 0001, 0002 suffixes)
                                    common_name_id = f'{root_folder}_{volunteer_folder}_{divo_folder}_slice_{i:03d}'

                                    # Save Average Diffusion Image as _0000
                                    # Save ONLY the selected contrasts (0=avg, 1=MD, 2=E1, 3=FA)
                                    _channels = {0: (avg_image_data, avg_img.affine),
                                                 1: (mean_diff_data, mean_diff_img.affine),
                                                 2: (combined_eigenvector_data, eigenvector_img.affine),
                                                 3: (FA_image_data, FA_image.affine)}
                                    for _ch in CONTRASTS:
                                        _arr, _aff = _channels[_ch]
                                        nib.save(nib.Nifti1Image(_arr, _aff),
                                                 os.path.join(output_image_folder, f'{common_name_id}_{_ch:04d}.nii.gz'))
                                    
                                

                                    # Save the mask
                                    nib.save(nib.Nifti1Image(combined_mask, mask_img.affine), os.path.join(output_mask_folder, f'{common_name_id}.nii.gz'))

                                    print(f'Saved mask slice {i}: {common_name_id}.nii.gz')
                                    print(f'Saved image modalities: {common_name_id}_0000.nii.gz, _0001.nii.gz, _0002.nii.gz')

                                    # Save inspection images (original, mask, overlay)
                                    save_inspection_plots(np.stack([avg_image_data, mean_diff_data, eigenvector_data[:, :, 1]], axis=-1), 
                                                          combined_mask, common_name_id)
                                else:
                                    print(f'Failed to find required files for slice {i}')
                        else:
                            # If any slice is not "Good Image", skip the processing and print the folder name
                            print(f'Skipping images, slice: {i} in volunter; {volunteer_folder} due to bad quality')
                    else:
                        print(f'Missing quality information in {divo_folder}')


In [ ]:
import json, os
with open(f'{OUTPUT_PWD}/{datasetname}/dataset.json', 'w') as f:
    json.dump({'channel_names': {str(c): 'noNorm' for c in range(len(CONTRASTS))},
               'labels': {'background': 0, 'LV': 1},
               'numTraining': len(os.listdir(output_mask_folder)),
               'file_ending': '.nii.gz'}, f, indent=4)
print(datasetname, '->', len(CONTRASTS), 'channels,', len(os.listdir(output_mask_folder)), 'cases')

## 2. IP — verbatim `NewFAIPOnly.py` + cohort pool + `CONTRASTS`

In [ ]:



import os
import glob
import re
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Root directory and subfolders
# ============ cohorts to POOL (comment out DirVsAverages for SmartHealth-only) ============
COHORTS = [
    ('/Users/saschastocker/Documents/Stanford/DanEnnis20242025/Paper2025Automatic/Smart_Health',  'Hannum'),
    ('/Users/saschastocker/Documents/Stanford/DanEnnis20242025/Paper2025Automatic/DirVsAverages', 'DirVsAveragesHannum'),
]
OUTPUT_PWD = '/Users/saschastocker/Documents/Stanford/DanEnnis20242025/Paper2025Automatic/Smart_Health'   # where the pooled dataset is written
CONTRASTS = [0, 1, 2, 3]   # 0=avg 1=MD 2=E1 3=FA  ->  use [0, 1] for the avg+MD MVP
datasetname = 'Dataset107_HannumSmartHealthDataandDirVsAvgIPs'
output_mask_folder = f'{OUTPUT_PWD}/{datasetname}/labelsTr'
output_image_folder = f'{OUTPUT_PWD}/{datasetname}/imagesTr'
inspection_folder = f'{OUTPUT_PWD}/inspection{datasetname}'

# Ensure output folders exist
os.makedirs(output_mask_folder, exist_ok=True)
os.makedirs(output_image_folder, exist_ok=True)
os.makedirs(inspection_folder, exist_ok=True)

def process_mask_slices(mask_data,lv_only ):
    """
    Combines three 2D mask slices (256, 256 each) into a single mask with:
    - LV (slice 1): 1
    - Insertion point (slice 2): 2
    - Second insertion point (slice 3): 3
    Output is a single 2D mask with shape (256, 256).
    """
    # Initialize the combined mask with zeros (256, 256)
    combined_mask = np.zeros((256, 256), dtype=np.uint8)
    if mask_data.shape[0] == 256 and mask_data.shape[-1] == 3:
        # Reshape to (3, 256, 256)
        mask_data = np.transpose(mask_data, (2, 0, 1))

    # Add slice 1 (LV) with label 1
    
    if(lv_only):
        combined_mask[mask_data[0, :, :] == 1] = 1


    #IPs only!   
    else:
         # Add slice 2 (Insertion point) with label 2
        combined_mask[mask_data[1, :, :] == 1] = 1

        # Add slice 3 (Second insertion point) with label 3
        combined_mask[mask_data[2, :, :] == 1] = 2

    return combined_mask

# Function to normalize images to [0, 1] range
def normalize_image(image):
    image_min = np.min(image)
    image_max = np.max(image)
    return (image - image_min) / (image_max - image_min)

def normalise_MD(image):
    image_min = 0
    image_max = 4
    return (image - image_min) / (image_max - image_min)

def normalise_eigenvector(image):
    # X and Y are components of a UNIT eigenvector (x^2+y^2+z^2=1), so their sum is
    # capped at sqrt(2) (~1.414), not 2 -- the unit-norm constraint couples them.
    # Divide by that fixed sqrt(2) ceiling instead of min-max: every image reaches
    # sqrt(2), whereas a true 0 (a purely through-plane voxel) exists in only ~46%
    # of images, so min-max would anchor on a noise-driven floor. Fixed = stable.
    return image / np.sqrt(2)

# Function to save images for inspection
def save_inspection_plots(image_data, mask_data, filename_base):
    """Saves inspection plots of image, mask, and overlay using matplotlib."""
    
    # Create a figure with 3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Plot 1: Just the image (first channel)
    axes[0].imshow(image_data[:, :, 0], cmap='gray')  # Show first channel (average)
    axes[0].set_title('Average Diffusion Image')
    axes[0].axis('off')

    # Plot 2: Just the mask
    axes[1].imshow(mask_data, cmap='gray')
    axes[1].set_title('Mask Only')
    axes[1].axis('off')

    # Plot 3: Image with mask overlay
    axes[2].imshow(image_data[:, :, 0], cmap='gray')  # Show first channel (average)
    mask_overlay = np.ma.masked_where(mask_data != 0, mask_data)
    axes[2].imshow(mask_overlay, cmap='Reds', alpha=0.5)
    axes[2].set_title('Image with Mask Overlay')
    axes[2].axis('off')

    # Save the figure
    output_file = os.path.join(inspection_folder, f'{filename_base}_inspection.png')
    plt.tight_layout()
    plt.savefig(output_file)
    plt.close(fig)
    print(f'Saved inspection plot: {output_file}')

for pwd, root_folder in COHORTS:
    print(f'root folder: {root_folder}')
    root_path = os.path.join(pwd, root_folder)

    # Loop through volunteer folders
    for volunteer_folder in os.listdir(root_path):
        if volunteer_folder.startswith('Volunteer'):
            volunteer_path = os.path.join(root_path, volunteer_folder)
            distortion_corrected_folder = os.path.join(volunteer_path, 'Distortion_Corrected')

            # Loop through DiVO and MDDW folders
            for divo_folder in os.listdir(distortion_corrected_folder):
                if divo_folder.startswith('DiVO') or divo_folder.startswith('MDDW'):
                    divo_path = os.path.join(distortion_corrected_folder, divo_folder)

                    # Load the quality control information from the Excel file in the DiVO/MDDW folder
                    excel_path = os.path.join(divo_path, 'Detailed_Information.xlsx')

                        # Check if the "Image Quality" column has "Good Image" for all rows (3 slices)
                        
                            # Proceed with the images only if all slices are of good quality

                    mask_slices = []
                    excel_path = os.path.join(divo_path, 'Detailed_Information.xlsx')
                    if os.path.exists(excel_path):
                        quality_data = pd.read_excel(excel_path)

                        mask_folder = os.path.join(divo_path, '06_Segmentation_Masks_CI')
                        image_folder = os.path.join(divo_path, '05_Segmentation_Images_CI')

                        # Count how many segmentation slices actually exist in this sub-folder
                        # (was hard-coded to the first 3 slices; folders can have many more, e.g. up to 12).
                        slice_files = glob.glob(os.path.join(mask_folder, 'Cropped_Segmentation_Slice_*.nii'))
                        slice_numbers = sorted(
                            int(re.search(r'Slice_(\d+)\.nii$', f).group(1)) for f in slice_files
                        )
                        num_slices = len(slice_numbers)
                        print(f'{volunteer_folder}/{divo_folder}: found {num_slices} slices -> {slice_numbers}')

                        # Loop through every slice that exists in this folder
                        for i in slice_numbers:
                            # Filter rows where "Slice Number" equals i
                            slice_data = quality_data[quality_data['Slice Number'] == i]

                            # Check if we found a matching row and if that row’s image quality is "Good Image Quality"
                            if not slice_data.empty and slice_data.iloc[0]['Image Quality'] == 'Good Image Quality':
                                mask_file = os.path.join(mask_folder, f'Cropped_Segmentation_Slice_{i:03d}.nii')
                                
                            
                                mask_img = nib.load(mask_file)

                                # Assuming the mask file is 3, 256, 256
                                mask_data = mask_img.get_fdata()  # Load the 3D mask (3, 256, 256)

                                # Process and combine the slices
                                #IPs only!!!
                                combined_mask = process_mask_slices(mask_data,lv_only=False)

                                
                                avg_image_file = os.path.join(image_folder, f'Cropped_Average_Diffusion_Weighted_Image_Slice_{i:03d}.nii')
                                mean_diff_file = os.path.join(image_folder, f'Cropped_Mean_Diffusivty_Image_Slice_{i:03d}.nii')
                                eigenvector_file = os.path.join(image_folder, f'Cropped_Primary_Eigenvector_Image_Slice_{i:03d}.nii')
                                FA_file = os.path.join(image_folder, f'Cropped_Fractional_Anisotropy_Image_Slice_{i:03d}.nii')

                                if os.path.exists(mask_file) and os.path.exists(avg_image_file) and os.path.exists(mean_diff_file) and os.path.exists(eigenvector_file):


                                    # Load the three NIfTI image files (average, mean diffusivity, eigenvector)
                                    avg_img = nib.load(avg_image_file)
                                    mean_diff_img = nib.load(mean_diff_file)
                                    eigenvector_img = nib.load(eigenvector_file)
                                    FA_image = nib.load(FA_file)

                                    # Get the data for all three images
                                    avg_image_data = avg_img.get_fdata()  # (256, 256)
                                    mean_diff_data = mean_diff_img.get_fdata()  # (256, 256)
                                    eigenvector_data = eigenvector_img.get_fdata()  # (256, 256, 3)
                                    FA_image_data = FA_image.get_fdata()

                                    # Normalize each image to [0, 1] range to prevent "washed out" effect
                                    avg_image_data = normalize_image(avg_image_data)
                                    mean_diff_data = normalise_MD(mean_diff_data)
                                    eigenvector_slice1 = eigenvector_data[:, :, 0]
                                    eigenvector_slice2 = eigenvector_data[:, :, 1]

                                    # Combine eigenvector slices 1 and 2 into a single slice
                                    combined_eigenvector_data = eigenvector_slice1 + eigenvector_slice2
                                    combined_eigenvector_data = normalise_eigenvector(combined_eigenvector_data)

                                    # Stack all three channels (average, mean diffusivity, combined eigenvector)
                                    combined_image_data = np.stack([avg_image_data, mean_diff_data, combined_eigenvector_data], axis=-1)

                                    # Save each channel separately (modality files with 0000, 0001, 0002 suffixes)
                                    common_name_id = f'{root_folder}_{volunteer_folder}_{divo_folder}_slice_{i:03d}'

                                    # Save Average Diffusion Image as _0000
                                    # Save ONLY the selected contrasts (0=avg, 1=MD, 2=E1, 3=FA)
                                    _channels = {0: (avg_image_data, avg_img.affine),
                                                 1: (mean_diff_data, mean_diff_img.affine),
                                                 2: (combined_eigenvector_data, eigenvector_img.affine),
                                                 3: (FA_image_data, FA_image.affine)}
                                    for _ch in CONTRASTS:
                                        _arr, _aff = _channels[_ch]
                                        nib.save(nib.Nifti1Image(_arr, _aff),
                                                 os.path.join(output_image_folder, f'{common_name_id}_{_ch:04d}.nii.gz'))
                                    
                                

                                    # Save the mask
                                    nib.save(nib.Nifti1Image(combined_mask, mask_img.affine), os.path.join(output_mask_folder, f'{common_name_id}.nii.gz'))

                                    print(f'Saved mask slice {i}: {common_name_id}.nii.gz')
                                    print(f'Saved image modalities: {common_name_id}_0000.nii.gz, _0001.nii.gz, _0002.nii.gz')

                                    # Save inspection images (original, mask, overlay)
                                    save_inspection_plots(np.stack([avg_image_data, mean_diff_data, eigenvector_data[:, :, 1]], axis=-1), 
                                                        combined_mask, common_name_id)
                                else:
                                    print(f'Failed to find required files for slice {i}')
                        else:
                            # If any slice is not "Good Image", skip the processing and print the folder name
                            print(f'Skipping images, slice: {i} in volunter; {volunteer_folder} due to bad quality')

                    else:
                        print(f'Missing quality information in {divo_folder}')


In [ ]:
import json, os
with open(f'{OUTPUT_PWD}/{datasetname}/dataset.json', 'w') as f:
    json.dump({'channel_names': {str(c): 'noNorm' for c in range(len(CONTRASTS))},
               'labels': {'background': 0, 'IP1': 1, 'IP2': 2},
               'numTraining': len(os.listdir(output_mask_folder)),
               'file_ending': '.nii.gz'}, f, indent=4)
print(datasetname, '->', len(CONTRASTS), 'channels,', len(os.listdir(output_mask_folder)), 'cases')

## 3. Crop — verbatim `NewCropPrep.py` + cohort pool (DWI only — already MVP)

In [ ]:
import os
import glob
import re
import nibabel as nib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Root directory and subfolders
# ============ cohorts to POOL (comment out DirVsAverages for SmartHealth-only) ============
COHORTS = [
    ('/Users/saschastocker/Documents/Stanford/DanEnnis20242025/Paper2025Automatic/Smart_Health',  'Hannum'),
    ('/Users/saschastocker/Documents/Stanford/DanEnnis20242025/Paper2025Automatic/DirVsAverages', 'DirVsAveragesHannum'),
]
OUTPUT_PWD = '/Users/saschastocker/Documents/Stanford/DanEnnis20242025/Paper2025Automatic/Smart_Health'   # where the pooled dataset is written
datasetname = 'Dataset111_HannumSmartHealthandDirVsAvgCrop'
output_mask_folder = f'{OUTPUT_PWD}/{datasetname}/labelsTr'
output_image_folder = f'{OUTPUT_PWD}/{datasetname}/imagesTr'
inspection_folder = f'{OUTPUT_PWD}/inspection{datasetname}'

# Ensure output folders existw
os.makedirs(output_mask_folder, exist_ok=True)
os.makedirs(output_image_folder, exist_ok=True)
os.makedirs(inspection_folder, exist_ok=True)

# Function to normalize images to [0, 1] range
def normalize_image(image):
    image_min = np.min(image)
    image_max = np.max(image)
    return (image - image_min) / (image_max - image_min)

# Function to save images for inspection
def save_inspection_plots(image_data, mask_data, filename_base):
    """Saves inspection plots of image, mask, and overlay using matplotlib."""
    
    # Create a figure with 3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Plot 1: Just the image
    axes[0].imshow(image_data, cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')

    # Plot 2: Just the mask
    axes[1].imshow(mask_data, cmap='gray')
    axes[1].set_title('Mask Only')
    axes[1].axis('off')

    # Plot 3: Image with mask overlay
    axes[2].imshow(image_data, cmap='gray')
    mask_overlay = np.ma.masked_where(mask_data != 1, mask_data)
    axes[2].imshow(mask_overlay, cmap='Reds', alpha=0.5)
    axes[2].set_title('Image with Mask Overlay')
    axes[2].axis('off')

    # Save the figure
    output_file = os.path.join(inspection_folder, f'{filename_base}_inspection.png')
    plt.tight_layout()
    plt.savefig(output_file)
    plt.close(fig)
    print(f'Saved inspection plot: {output_file}')


for pwd, root_folder in COHORTS:
    print(f'root folder: {root_folder}')
    root_path = os.path.join(pwd, root_folder)

    # Loop through volunteer folders
    for volunteer_folder in os.listdir(root_path):
        if volunteer_folder.startswith('Volunteer'):
            volunteer_path = os.path.join(root_path, volunteer_folder)
            distortion_corrected_folder = os.path.join(volunteer_path, 'Distortion_Corrected')

            # Loop through DiVO and MDDW folders
            for divo_folder in os.listdir(distortion_corrected_folder):
                if divo_folder.startswith('DiVO') or divo_folder.startswith('MDDW'):
                    divo_path = os.path.join(distortion_corrected_folder, divo_folder)

                    # Load the quality control information from the Excel file in the DiVO/MDDW folder
                    excel_path = os.path.join(divo_path, 'Detailed_Information.xlsx')
                    if os.path.exists(excel_path):
                        quality_data = pd.read_excel(excel_path)

                        mask_folder = os.path.join(divo_path, '02_Crop_Masks')
                        image_folder = os.path.join(divo_path, '03_Segmentation_Images')

                        # Count how many crop-mask slices actually exist in this sub-folder
                        # (was hard-coded to the first 3 slices; folders can have many more, e.g. up to 12).
                        slice_files = glob.glob(os.path.join(mask_folder, 'Square_Crop_Mask_Slice_*.nii'))
                        slice_numbers = sorted(
                            int(re.search(r'Slice_(\d+)\.nii$', f).group(1)) for f in slice_files
                        )
                        num_slices = len(slice_numbers)
                        print(f'{volunteer_folder}/{divo_folder}: found {num_slices} slices -> {slice_numbers}')

                        # Loop through every slice that exists in this folder
                        for i in slice_numbers:
                            # Filter rows where "Slice Number" equals i
                            slice_data = quality_data[quality_data['Slice Number'] == i]
    
                            # Check if we found a matching row and if that row’s image quality is "Good Image Quality"
                            if not slice_data.empty and slice_data.iloc[0]['Image Quality'] == 'Good Image Quality':

                            
                                mask_folder = os.path.join(divo_path, '02_Crop_Masks')
                                image_folder = os.path.join(divo_path, '03_Segmentation_Images')
                                mask_folder = os.path.join(divo_path, '02_Crop_Masks')
                                image_folder = os.path.join(divo_path, '03_Segmentation_Images')
                                # Select mask and image files for each iteration
                                mask_file = os.path.join(mask_folder, f'Square_Crop_Mask_Slice_{i:03d}.nii')
                                image_file = os.path.join(image_folder, f'Average_Diffusion_Weighted_Image_Slice_{i:03d}.nii')

        
                                if os.path.exists(mask_file) and os.path.exists(image_file):
                                    # Load the NIfTI mask file and extract the 0th slice
                                    mask_img = nib.load(mask_file)
                                    mask_data = mask_img.get_fdata()

                                    # Load the NIfTI image file (no slicing needed)
                                    image_img = nib.load(image_file)
                                    image_data = image_img.get_fdata()
                                    image_data = normalize_image(image_data)

                                    common_name_id = f'{root_folder}_{volunteer_folder}_{divo_folder}_slice_{i:03d}'

                                    mask_output_filename = os.path.join(output_mask_folder,
                                                                        f'{common_name_id}.nii.gz') 
                                    image_output_filename = os.path.join(output_image_folder,
                                                                        f'{common_name_id}_0000.nii.gz')

                                    # Save the mask and image
                                    nib.save(nib.Nifti1Image(mask_data, mask_img.affine), mask_output_filename)
                                    nib.save(nib.Nifti1Image(image_data, image_img.affine), image_output_filename)

                                    print(f'Saved mask slice {i}: {mask_output_filename}')
                                    print(f'Saved image slice {i}: {image_output_filename}')

                                    # Save inspection images (original, mask, overlay)
                                
                                    save_inspection_plots(image_data, mask_data, common_name_id)
                                else: 
                                    if not os.path.exists(mask_file):
                                        print(f'Failed to find mask file: {mask_file}')
                                    if not os.path.exists(image_file):
                                        print(f'Failed to find image file: {image_file}')
                            else:
                                print(f' bad image quality, skipping')
                                print(f'Skipping images, slice: {i} in volunter; {volunteer_folder} due to bad quality')
                           
                    else:
                        print(f'Missing quality information in {divo_folder}')


In [ ]:
import json, os
with open(f'{OUTPUT_PWD}/{datasetname}/dataset.json', 'w') as f:
    json.dump({'channel_names': {'0': 'noNorm'},
               'labels': {'background': 0, 'crop': 1},
               'numTraining': len(os.listdir(output_mask_folder)),
               'file_ending': '.nii.gz'}, f, indent=4)
print(datasetname, '->', len(os.listdir(output_mask_folder)), 'cases')